In [0]:
# Camada Silver - Limpeza, tipagem, padronização e deduplicação dos dados

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "workspace"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

In [0]:
def ler_ultimo_lote_bronze(nome_tabela):
    df_bronze = spark.read.table(f"{CATALOGO}.{SCHEMA_BRONZE}.{nome_tabela}")
    ultimo_lote = df_bronze.agg(F.max("ingestion_datetime")).collect()[0][0]
    return df_bronze.filter(F.col("ingestion_datetime") == ultimo_lote)

In [0]:
def salvar_silver(df, nome_tabela):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(f"{CATALOGO}.{SCHEMA_SILVER}.{nome_tabela}")
    )

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.silver;

## silver.tb_info_filmes

In [0]:
# Leitura:
df_bronze = ler_ultimo_lote_bronze("tb_movies_info")

# Normalização
status_normalizado = F.trim(
    F.regexp_replace(
        F.regexp_replace(
            F.regexp_replace(F.lower(F.col("status")), r"[-_]+", " "),
            r"[^a-z ]", ""),
        r"\s+", " ")
)

mapa_status = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado",
}

# Valores não mapeáveis
status_traduzido = F.lit("Não Informado")
for status_origem, status_destino in mapa_status.items():
    status_traduzido = F.when(status_normalizado == status_origem, status_destino).otherwise(status_traduzido)

# Tratameno da data multi-formato
data_lancamento = F.coalesce(
    F.to_date(F.expr("try_to_timestamp(trim(release_date), 'yyyy-MM-dd')")),
    F.to_date(F.expr("try_to_timestamp(trim(release_date), 'dd/MM/yyyy')")),
    F.to_date(F.expr("try_to_timestamp(trim(release_date), 'MM-dd-yyyy')")),
)

df_tratado = (
    df_bronze
    .select(
        F.trim(F.col("id")).alias("id_filme"),
        F.col("title").alias("titulo"),
        F.col("original_title").alias("titulo_original"),
        data_lancamento.alias("data_lancamento"),
        F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"),
        F.col("original_language").alias("idioma_original"),
        status_traduzido.alias("status_filme"),
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("frase_divulgacao"),
        F.col("ingestion_datetime"),
    )
    .withColumn("ano_lancamento", F.year("data_lancamento"))
)

# Deduplicação
janela_filme = Window.partitionBy("id_filme").orderBy(
    F.col("ingestion_datetime").desc(),
    F.col("sinopse").isNull(),
)

df_silver = (
    df_tratado
    .withColumn("ordem", F.row_number().over(janela_filme))
    .filter(F.col("ordem") == 1)
    .drop("ordem", "ingestion_datetime")
)

display(df_silver.limit(10))


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000004,Purple Beatz,Purple Beatz,2022-07-07,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,2022
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null,2020
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null,2022
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null,2022
1000014,On va manquer !,On va manquer !,2018-05-15,0,fr,Lançado,null,null,2018
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,0,es,Lançado,null,null,2021
1000054,One Hundred Years and Hope,百年と希望,2022-06-18,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null,2022
1000058,Homecoming,Le retour,2023-07-12,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances.",null,2023
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null,2016
1000073,A Chance To Win,Pour l'honneur,2023-05-03,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null,2023


In [0]:
# Validação
print(f"Registros: {df_silver.count()} | ids distintos: {df_silver.select('id_filme').distinct().count()}")
print(f"Datas não convertidas: {df_silver.filter(F.col('data_lancamento').isNull()).count()}")
display(df_silver.groupBy("status_filme").count().orderBy(F.desc("count")))

# Escrita:
salvar_silver(df_silver, "tb_info_filmes")

Registros: 97611 | ids distintos: 97611
Datas não convertidas: 2


status_filme,count
Lançado,96261
Pós-Produção,697
Em Produção,604
Planejado,47
Não Informado,2


##  silver.tb_financeiro_filmes

In [0]:
def limpar_valor_monetario(df, coluna_origem, coluna_destino):
    texto = F.upper(F.trim(F.col(coluna_origem)))
    texto_ausente = texto.isin("UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "N/A", "")

    multiplicador = (
        F.when(texto.endswith("M"), F.lit(1000000))
        .when(texto.endswith("K"), F.lit(1000))
        .otherwise(F.lit(1))
    )

    # com sufixo => o ponto é separador decimal e é mantido
    # sem sufixo => pontos e vírgulas são pontuação de milhar e são removidos
    numero_texto = (
        F.when(texto_ausente, F.lit(None))
        .when(texto.endswith("M") | texto.endswith("K"), F.regexp_replace(texto, r"[^0-9.\-]", ""))
        .otherwise(F.regexp_replace(texto, r"[^0-9\-]", ""))
    )

    return (
        df
        .withColumn("_numero_texto", numero_texto)
        .withColumn("_multiplicador", multiplicador)
        .withColumn(
            coluna_destino,
            (F.expr("try_cast(_numero_texto AS DECIMAL(18,2))") * F.col("_multiplicador")).cast("decimal(18,2)"),
        )
        .withColumn(coluna_destino, F.when(F.col(coluna_destino) > 0, F.col(coluna_destino)))
        .drop("_numero_texto", "_multiplicador")
    )

In [0]:
df_bronze = ler_ultimo_lote_bronze("tb_movies_financials")

# Taxa de conversão
linha_cotacao = (
    spark.read.table(f"{CATALOGO}.{SCHEMA_BRONZE}.tb_cotacao_dolar")
    .select(
        F.to_date(F.substring("dataHoraCotacao", 1, 10)).alias("data_cotacao"),
        F.col("cotacaoCompra").cast("decimal(10,4)").alias("cotacao_compra"),
    )
    .orderBy(F.col("data_cotacao").desc())
    .first()
)
cotacao_dolar = linha_cotacao["cotacao_compra"]
data_cotacao = linha_cotacao["data_cotacao"]
print(f"Cotação utilizada: R$ {cotacao_dolar} ({data_cotacao})")

df_limpo = df_bronze.select(F.trim(F.col("id")).alias("id_filme"), "budget", "revenue")
df_limpo = limpar_valor_monetario(df_limpo, "budget", "orcamento_usd")
df_limpo = limpar_valor_monetario(df_limpo, "revenue", "receita_usd")

# Consolidação por filme
df_consolidado = (
    df_limpo
    .groupBy("id_filme")
    .agg(
        F.max("orcamento_usd").alias("orcamento_usd"),
        F.max("receita_usd").alias("receita_usd"),
    )
)

# Conversão para BRL e métricas derivadas
df_silver = (
    df_consolidado
    .withColumn("cotacao_dolar", F.lit(cotacao_dolar).cast("decimal(10,4)"))
    .withColumn("data_cotacao", F.lit(data_cotacao).cast("date"))
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("cotacao_dolar")).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.col("cotacao_dolar")).cast("decimal(18,2)"))
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    .withColumn(
        "margem_lucro_pct",
        F.when(
            F.col("receita_usd") > 0,
            F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2),
        ).cast("decimal(10,2)"),
    )
)

display(df_silver.orderBy(F.desc("receita_usd")).limit(10))

Cotação utilizada: R$ 4.9929 (2026-10-07)


id_filme,orcamento_usd,receita_usd,cotacao_dolar,data_cotacao,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
299534,356000000.00,2800000000.00,4.9929,2026-10-07,1777472400.00,13980120000.00,2444000000.00,12202647600.00,87.29
76600,460000000.00,2320250281.00,4.9929,2026-10-07,2296734000.00,11584777628.00,1860250281.00,9288043628.00,80.17
299536,300000000.00,2052415039.00,4.9929,2026-10-07,1497870000.00,10247503048.22,1752415039.00,8749633048.22,85.38
634649,200000000.00,1921847111.00,4.9929,2026-10-07,998580000.00,9595590440.51,1721847111.00,8597010440.51,89.59
420818,260000000.00,1663075401.00,4.9929,2026-10-07,1298154000.00,8303569169.65,1403075401.00,7005415169.65,84.37
361743,170000000.00,1488732821.00,4.9929,2026-10-07,848793000.00,7433094101.97,1318732821.00,6584301101.97,88.58
346698,145000000.00,1428545028.00,4.9929,2026-10-07,723970500.00,7132582470.30,1283545028.00,6408611970.30,89.85
502356,100000000.00,1355725263.00,4.9929,2026-10-07,499290000.00,6769000665.63,1255725263.00,6269710665.63,92.62
284054,200000000.00,1349926083.00,4.9929,2026-10-07,998580000.00,6740045939.81,1149926083.00,5741465939.81,85.18
181808,200000000.00,1332698830.00,4.9929,2026-10-07,998580000.00,6654031988.31,1132698830.00,5655451988.31,84.99


In [0]:
print(f"Registros: {df_silver.count()} | ids distintos: {df_silver.select('id_filme').distinct().count()}")
display(
    df_silver.select(
        F.count(F.when(F.col("orcamento_usd").isNull(), 1)).alias("orcamento_nulo"),
        F.count(F.when(F.col("receita_usd").isNull(), 1)).alias("receita_nula"),
        F.count(F.when(F.col("lucro_usd").isNotNull(), 1)).alias("com_lucro_calculado"),
    )
)

salvar_silver(df_silver, "tb_financeiro_filmes")

Registros: 99006 | ids distintos: 99006


orcamento_nulo,receita_nula,com_lucro_calculado
90813,95707,1577


## silver.tb_metricas_engajamento


In [0]:
df_bronze = ler_ultimo_lote_bronze("tb_movies_metrics")

PADRAO_DECIMAL = r"^-?\d+(\.\d+)?$"
PADRAO_INTEIRO = r"^-?\d+$"

def para_decimal(coluna):
    texto = F.trim(F.col(coluna))
    return F.when(texto.rlike(PADRAO_DECIMAL), texto).cast("double")


def para_inteiro(coluna):
    texto = F.trim(F.col(coluna))
    valor = F.when(texto.rlike(PADRAO_INTEIRO), texto).cast("double")
    return F.when(valor.between(0, 2147483647), valor).cast("int")


# Padronização da vírgula na popularidade
popularidade_texto = F.regexp_replace(F.trim(F.col("popularity")), ",", ".")

df_tratado = (
    df_bronze
    .select(
        F.trim(F.col("id")).alias("id_filme"),
        F.when(popularidade_texto.rlike(PADRAO_DECIMAL), popularidade_texto).cast("double").alias("popularidade"),
        para_decimal("vote_average").alias("nota_media_tmdb"),
        para_inteiro("vote_count").alias("qtd_votos_tmdb"),
        para_decimal("averageRating").alias("nota_media_imdb"),
        para_inteiro("numVotes").alias("qtd_votos_imdb"),
    )
    # Regras de negócio:
    # - Notas fora de 0 a 10 (inclui erro de escala, ex.: 65.5) -> NULL
    # - Popularidade negativa -> NULL (votos negativos já são barrados em para_inteiro)
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
)

# Consolidação por filme
df_silver = (
    df_tratado
    .groupBy("id_filme")
    .agg(
        F.max("popularidade").alias("popularidade"),
        F.max("nota_media_tmdb").alias("nota_media_tmdb"),
        F.max("qtd_votos_tmdb").alias("qtd_votos_tmdb"),
        F.max("nota_media_imdb").alias("nota_media_imdb"),
        F.max("qtd_votos_imdb").alias("qtd_votos_imdb"),
    )
)

display(df_silver.orderBy(F.desc("popularidade")).limit(10))

# Escrita:
salvar_silver(df_silver, "tb_metricas_engajamento")

id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
565770,2994.357,7.139,1023,5.9,119501
980489,2680.593,8.068,null,7.1,138696
809905,2020.0,5.0,2,null,null
658829,2019.0,5.4,5,null,null
418746,1969.0,10.0,null,null,null
968051,1692.778,6.545,365,5.6,84018
615656,1567.273,6.912,2034,null,105031
762430,1547.22,6.816,258,null,29152
1008042,1458.514,7.214,973,7.1,224148
385687,1175.267,null,3881,5.7,149785


## silver.tb_avaliacoes_usuarios

In [0]:
df_bronze = ler_ultimo_lote_bronze("tb_movies_reviews")

nota_texto = F.trim(F.col("nota"))

df_silver = (
    df_bronze
    .select(
        F.trim(F.col("id")).alias("id_filme"),
        F.trim(F.col("nome")).alias("nome_usuario"),
        F.when(nota_texto.rlike(PADRAO_DECIMAL), nota_texto).cast("double").alias("nota_usuario"),
        F.col("comentario").alias("comentario_usuario"),
    )
    # nota fora da escala 0 a 10 -> NULL
    .withColumn("nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")))
    .withColumn(
        "comentario_usuario",
        F.when(
            F.col("comentario_usuario").isNull() | (F.trim(F.col("comentario_usuario")) == ""),
            F.lit("Sem comentário"),
        ).otherwise(F.trim(F.col("comentario_usuario"))),
    )
    # Deduplicação feita APÓS a padronização
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)

display(df_silver.limit(10))

# Escrita:
salvar_silver(df_silver, "tb_avaliacoes_usuarios")

id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
637007,Lucas Reis 602,3.9,Sem comentário
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,Sem comentário
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


## silver.tb_generos

In [0]:
def explodir_coluna(df, coluna_origem):
    return (
        df.select(
            F.trim(F.col("id")).alias("id_filme"),
            F.explode(F.split(F.regexp_replace(F.col(coluna_origem), r"[;|]", ","), ",")).alias("valor"),
        )
        .withColumn("valor", F.trim(F.col("valor")))
    )


GENEROS_VALIDOS = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "TV Movie", "Thriller", "War", "Western",
]

df_bronze_credits = ler_ultimo_lote_bronze("tb_credits_and_tags")

df_silver = (
    explodir_coluna(df_bronze_credits, "genres")
    .filter(F.col("valor").isin(GENEROS_VALIDOS))
    .select("id_filme", F.col("valor").alias("nome_genero"))
    .dropDuplicates(["id_filme", "nome_genero"])
)

display(df_silver.groupBy("nome_genero").count().orderBy(F.desc("count")))

salvar_silver(df_silver, "tb_generos")

nome_genero,count
Drama,32615
Documentary,19235
Comedy,18827
Thriller,10400
Horror,9833
Romance,7717
Action,6114
Crime,4786
Animation,4524
TV Movie,4115


## silver.tb_pessoas_empresas

In [0]:
# Mapeamento coluna de origem => tipo de entidade
mapa_entidades = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora",
}

# Unifica as 4 colunas em um único modelo
df_entidades = None
for coluna_origem, tipo_entidade in mapa_entidades.items():
    df_tipo = explodir_coluna(df_bronze_credits, coluna_origem).withColumn("tipo_entidade", F.lit(tipo_entidade))
    df_entidades = df_tipo if df_entidades is None else df_entidades.unionByName(df_tipo)

# Valores de outros domínios que aparecem deslocados nas colunas de entidades
df_outros_dominios = (
    explodir_coluna(df_bronze_credits, "production_countries")
    .unionByName(explodir_coluna(df_bronze_credits, "spoken_languages"))
    .groupBy("valor").count()
    .filter(F.col("count") >= 50)
    .select("valor")
    .unionByName(spark.createDataFrame([(g,) for g in GENEROS_VALIDOS], ["valor"]))
    .distinct()
)

VALORES_AUSENTES = ["", "[]", "N/A", "Nenhum", "None", "Unknown", "Não Informado"]

df_silver = (
    df_entidades
    .filter(~F.col("valor").isin(VALORES_AUSENTES))       
    .filter(F.col("valor").rlike(r"\p{L}"))               
    .filter(~F.col("valor").startswith("/"))              
    .filter(~F.col("valor").rlike(r'["\\]'))              
    .filter(~F.col("valor").rlike(r"^[a-z0-9 '\-]+$"))    
    .filter(F.length("valor") <= 80)                      
    .join(df_outros_dominios, "valor", "left_anti")       
    .withColumn("nome_entidade", F.initcap(F.lower(F.col("valor"))))
    .select("id_filme", "nome_entidade", "tipo_entidade")
    .dropDuplicates()
)

display(df_silver.groupBy("tipo_entidade").count())

# Escrita:
salvar_silver(df_silver, "tb_pessoas_empresas")

tipo_entidade,count
Roteirista,124921
Diretor,94520
Ator,545652
Produtora,116522


In [0]:
df_bronze = spark.read.table(f"{CATALOGO}.{SCHEMA_BRONZE}.tb_cotacao_dolar")

# Uma cotação por dia
janela_dia = Window.partitionBy("data_cotacao").orderBy(F.col("ingestion_datetime").desc())

df_cotacoes = (
    df_bronze
    .select(
        F.to_date(F.substring("dataHoraCotacao", 1, 10)).alias("data_cotacao"),
        F.col("cotacaoCompra").cast("decimal(10,4)").alias("cotacao_compra"),
        F.col("ingestion_datetime"),
    )
    .withColumn("ordem", F.row_number().over(janela_dia))
    .filter(F.col("ordem") == 1)
    .select("data_cotacao", "cotacao_compra")
)

# Calendário contínuo
df_calendario = (
    df_cotacoes
    .agg(F.min("data_cotacao").alias("data_inicio"))
    .select(F.explode(F.sequence(F.col("data_inicio"), F.current_date())).alias("data_cotacao"))
)

# Forward fill
janela_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)

df_silver = (
    df_calendario
    .join(df_cotacoes, "data_cotacao", "left")
    # Flag de rastreabilidade: indica se o valor foi preenchido (não veio da API)
    .withColumn("cotacao_preenchida", F.col("cotacao_compra").isNull())
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(janela_ffill))
    .orderBy("data_cotacao")
)

display(df_silver)

# Escrita:
salvar_silver(df_silver, "tb_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra,cotacao_preenchida
2026-09-30,5.1803,false
2026-10-01,5.2073,false
2026-10-02,5.2232,false
2026-10-03,5.2232,true
2026-10-04,5.2232,true
2026-10-05,4.9853,false
2026-10-06,4.9692,false
2026-10-07,4.9929,false


In [0]:
# Validação final

tabelas_silver = [
    "tb_info_filmes", "tb_financeiro_filmes", "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios", "tb_generos", "tb_pessoas_empresas", "tb_cotacao_dolar",
]

for nome_tabela in tabelas_silver:
    df_check = spark.read.table(f"{CATALOGO}.{SCHEMA_SILVER}.{nome_tabela}")
    print(f"\n{nome_tabela}: {df_check.count()} registros")
    df_check.printSchema()


tb_info_filmes: 97611 registros
root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)


tb_financeiro_filmes: 99006 registros
root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- cotacao_dolar: decimal(10,4) (nullable = true)
 |-- data_cotacao: date (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_pct: decimal(10,2) (nu